In [4]:
import pandas as pd
import shutil

# Preserve an unchanged copy of the original dataset
shutil.copy(r"C:\Users\Mchow\Documents\health-insurance\insurance.csv", r"C:\Users\Mchow\Documents\health-insurance\insurance_raw.csv")

# Load the dataset
df = pd.read_csv(
    r"C:\Users\Mchow\Documents\health-insurance\insurance.csv"
)

print(df.head())

   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


In [8]:
df.shape

(1338, 7)

In [3]:
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


In [9]:
df.dtypes

age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object

In [10]:
df.isnull().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [5]:
categorical_columns = df.select_dtypes(
    include=['object', 'category']
).columns

for col in categorical_columns:
    print(col)
    print(df[col].value_counts())

sex
sex
male      676
female    662
Name: count, dtype: int64
smoker
smoker
no     1064
yes     274
Name: count, dtype: int64
region
region
southeast    364
southwest    325
northwest    325
northeast    324
Name: count, dtype: int64


In [11]:
# Check missing values
print("Missing values:")
print(df.isnull().sum())

print("\nTotal missing values:")
print(df.isnull().sum().sum())

Missing values:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

Total missing values:
0


In [12]:
# Count duplicate records
print("Duplicate rows:", df.duplicated().sum())

# Display duplicate records if present
df[df.duplicated(keep=False)]

Duplicate rows: 1


,age,sex,bmi,children,smoker,region,charges
195,19,male,30.59,0,no,northwest,1639.5631
581,19,male,30.59,0,no,northwest,1639.5631


In [13]:
# Examine categorical columns
for col in ['sex', 'smoker', 'region']:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))


sex:
sex
male      676
female    662
Name: count, dtype: int64

smoker:
smoker
no     1064
yes     274
Name: count, dtype: int64

region:
region
southeast    364
southwest    325
northwest    325
northeast    324
Name: count, dtype: int64


In [14]:
for col in ['sex', 'smoker', 'region']:
    print(col, df[col].unique())

sex ['female' 'male']
smoker ['yes' 'no']
region ['southwest' 'southeast' 'northwest' 'northeast']


In [15]:
for col in ['sex', 'smoker', 'region']:
    df[col] = df[col].str.strip().str.lower()

In [16]:
print(df.dtypes)

age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
dtype: object


In [17]:
print("Age below 18 or above 100:")
print(df[~df['age'].between(18, 100)])

print("BMI below 10 or above 80:")
print(df[~df['bmi'].between(10, 80)])

print("Negative children:")
print(df[df['children'] < 0])

print("Zero or negative charges:")
print(df[df['charges'] <= 0])

Age below 18 or above 100:
Empty DataFrame
Columns: [age, sex, bmi, children, smoker, region, charges]
Index: []
BMI below 10 or above 80:
Empty DataFrame
Columns: [age, sex, bmi, children, smoker, region, charges]
Index: []
Negative children:
Empty DataFrame
Columns: [age, sex, bmi, children, smoker, region, charges]
Index: []
Zero or negative charges:
Empty DataFrame
Columns: [age, sex, bmi, children, smoker, region, charges]
Index: []


In [18]:
numeric_columns = ['age', 'bmi', 'children', 'charges']

for col in numeric_columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower) | (df[col] > upper)
    ]

    print(f"\nOutliers in {col}:")
    print(len(outliers))


Outliers in age:
0

Outliers in bmi:
9

Outliers in children:
0

Outliers in charges:
139


In [19]:
df['age_band'] = pd.cut(
    df['age'],
    bins=[17, 25, 35, 45, 55, 65, float('inf')],
    labels=['18-25', '26-35', '36-45',
            '46-55', '56-65', '66+']
)

print(df[['age', 'age_band']].head(10))

   age age_band
0   19    18-25
1   18    18-25
2   28    26-35
3   33    26-35
4   32    26-35
5   31    26-35
6   46    46-55
7   37    36-45
8   37    36-45
9   60    56-65


In [20]:
print("Dataset dimensions:", df.shape)

print("\nNew age-band distribution:")
print(df['age_band'].value_counts(sort=False))

print("\nFinal data types:")
print(df.dtypes)

print("\nFinal missing values:")
print(df.isnull().sum())

Dataset dimensions: (1338, 8)

New age-band distribution:
age_band
18-25    306
26-35    268
36-45    264
46-55    284
56-65    216
66+        0
Name: count, dtype: int64

Final data types:
age            int64
sex           object
bmi          float64
children       int64
smoker        object
region        object
charges      float64
age_band    category
dtype: object

Final missing values:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
age_band    0
dtype: int64


In [21]:
df.to_csv(
    r"C:\Users\Mchow\Documents\health-insurance\insurance_transformed.csv",
    index=False
)

In [22]:
import pandas as pd
import shutil

# Load original dataset
df_raw = pd.read_csv(
    r"C:\Users\Mchow\Documents\health-insurance\insurance.csv"
)

# Create a separate working DataFrame
df_cleaned = df_raw.copy(deep=True)

print("Original dataset:", df_raw.shape)
print("Working dataset:", df_cleaned.shape)

Original dataset: (1338, 7)
Working dataset: (1338, 7)


In [23]:
# Standardize categorical values
categorical_columns = ['sex', 'smoker', 'region']

for col in categorical_columns:
    df_cleaned[col] = (
        df_cleaned[col].str.strip().str.lower()
    )

# Check and remove exact duplicate rows
duplicates = df_cleaned.duplicated().sum()

print("Duplicate rows found:", duplicates)

# Remove exact duplicates if present
df_cleaned = df_cleaned.drop_duplicates()

# Create age bands
df_cleaned['age_band'] = pd.cut(
    df_cleaned['age'],
    bins=[17, 25, 35, 45, 55, 65, float('inf')],
    labels=['18-25', '26-35', '36-45',
            '46-55', '56-65', '66+']
)

print("Cleaned dataset shape:", df_cleaned.shape)

Duplicate rows found: 1
Cleaned dataset shape: (1337, 8)


In [24]:
cleaned_path = (
    r"C:\Users\Mchow\Documents\health-insurance\insurance_cleaned.csv"
)

df_cleaned.to_csv(cleaned_path, index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [25]:
df_cleaned_reloaded = pd.read_csv(cleaned_path)

print(df_cleaned_reloaded.head())

   age     sex     bmi  children smoker     region      charges age_band
0   19  female  27.900         0    yes  southwest  16884.92400    18-25
1   18    male  33.770         1     no  southeast   1725.55230    18-25
2   28    male  33.000         3     no  southeast   4449.46200    26-35
3   33    male  22.705         0     no  northwest  21984.47061    26-35
4   32    male  28.880         0     no  northwest   3866.85520    26-35


In [26]:
print("Dataset dimensions:")
print(df_cleaned_reloaded.shape)

print("\nData types:")
print(df_cleaned_reloaded.dtypes)

Dataset dimensions:
(1337, 8)

Data types:
age           int64
sex          object
bmi         float64
children      int64
smoker       object
region       object
charges     float64
age_band     object
dtype: object


In [27]:
print("Missing values:")
print(df_cleaned_reloaded.isnull().sum())

print("\nCategory counts:")

for col in ['sex', 'smoker', 'region', 'age_band']:
    print(f"\n{col}:")
    print(df_cleaned_reloaded[col].value_counts(dropna=False))

Missing values:
age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
age_band    0
dtype: int64

Category counts:

sex:
sex
male      675
female    662
Name: count, dtype: int64

smoker:
smoker
no     1063
yes     274
Name: count, dtype: int64

region:
region
southeast    364
southwest    325
northwest    324
northeast    324
Name: count, dtype: int64

age_band:
age_band
18-25    305
46-55    284
26-35    268
36-45    264
56-65    216
Name: count, dtype: int64


In [28]:
print("Numerical summaries:")
print(df_cleaned_reloaded.describe())

print("\nInsurance charges by smoker:")
print(
    df_cleaned_reloaded.groupby('smoker')['charges']
    .agg(['count', 'mean', 'median', 'min', 'max'])
)

print("\nInsurance charges by age band:")
print(
    df_cleaned_reloaded.groupby('age_band', observed=False)['charges']
    .agg(['count', 'mean', 'median'])
)

Numerical summaries:
               age          bmi     children       charges
count  1337.000000  1337.000000  1337.000000   1337.000000
mean     39.222139    30.663452     1.095737  13279.121487
std      14.044333     6.100468     1.205571  12110.359656
min      18.000000    15.960000     0.000000   1121.873900
25%      27.000000    26.290000     0.000000   4746.344000
50%      39.000000    30.400000     1.000000   9386.161300
75%      51.000000    34.700000     2.000000  16657.717450
max      64.000000    53.130000     5.000000  63770.428010

Insurance charges by smoker:
        count          mean       median         min          max
smoker                                                           
no       1063   8440.660307   7345.72660   1121.8739  36910.60803
yes       274  32050.231832  34456.34845  12829.4551  63770.42801

Insurance charges by age band:
          count          mean        median
age_band                                   
18-25       305   9111.433685   26

In [29]:
change_log = pd.DataFrame({
    'Operation': [
        'Loaded original dataset',
        'Created working copy',
        'Standardized categorical values',
        'Checked duplicate records',
        'Removed exact duplicates if present',
        'Created age_band variable',
        'Preserved high insurance charges',
        'Saved cleaned CSV',
        'Reloaded cleaned CSV'
    ],
    'Reason': [
        'Read original CSV using pandas',
        'Protected original dataset',
        'Ensure consistent category formatting',
        'Identify repeated observations',
        'Avoid accidental duplicate records',
        'Support age-group comparisons',
        'Retain valid observations',
        'Create reusable cleaned dataset',
        'Verify saved dataset'
    ]
})

print(change_log.to_string(index=False))

                          Operation                                Reason
            Loaded original dataset        Read original CSV using pandas
               Created working copy            Protected original dataset
    Standardized categorical values Ensure consistent category formatting
          Checked duplicate records        Identify repeated observations
Removed exact duplicates if present    Avoid accidental duplicate records
          Created age_band variable         Support age-group comparisons
   Preserved high insurance charges             Retain valid observations
                  Saved cleaned CSV       Create reusable cleaned dataset
               Reloaded cleaned CSV                  Verify saved dataset


In [30]:
change_log.to_csv(
    r"C:\Users\Mchow\Documents\health-insurance\cleaning_log.csv",
    index=False
)

**Load and Validate cleaned dataset**

In [8]:
from pathlib import Path
import pandas as pd

def load_dataset(file_path):
    """Load a CSV dataset and check that it exists."""

    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"Dataset not found: {file_path}"
        )

    return pd.read_csv(file_path)


def validate_dataset(df):
    """Check the expected dataset structure."""

    expected_columns = [
        "age", "sex", "bmi", "children",
        "smoker", "region", "charges"
    ]

    missing_columns = set(expected_columns) - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing columns: {missing_columns}"
        )

    if df.empty:
        raise ValueError("Dataset is empty.")

    return True


# Load and validate the cleaned dataset
df = load_dataset(r"C:\Users\Mchow\Documents\health-insurance\insurance_cleaned.csv")

validate_dataset(df)

print("Dataset loaded and validated successfully.")


Dataset loaded and validated successfully.
